In [1]:
import pandas as pd
import numpy as np
import joblib
import shap

In [2]:
df = pd.read_csv("../data/ascentra_academic_risk.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (4424, 16)


,student_id,Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Admission grade,attendance_percentage,ct1_score,ct2_score,assignment_average,quiz_average,lms_login_frequency,lms_resource_access,late_submission_count,academic_trend,academic_risk
0,STU0001,0,0.000000,0,0,127.3,54.4,10.4,52.9,40.0,38.9,1.1,20.2,7,Declining,1.0
1,STU0002,6,14.000000,6,6,142.5,72.5,72.9,71.2,51.0,55.1,4.4,55.7,3,Stable,0.0
2,STU0003,0,0.000000,6,0,124.8,64.7,51.2,50.2,58.0,64.5,3.5,18.7,8,Declining,1.0
3,STU0004,6,13.428571,6,8,119.6,53.6,67.6,56.7,61.3,69.6,5.7,69.6,4,Stable,0.0
4,STU0005,5,12.333333,6,9,141.5,61.8,52.1,51.7,68.4,40.1,1.3,55.8,4,Stable,0.0


In [3]:
xgb_ascentra = joblib.load(
    "../models/ascentra_xgboost.pkl"
)

preprocessor = joblib.load(
    "../models/ascentra_preprocessor.pkl"
)

print("Model and preprocessor loaded successfully!")

Model and preprocessor loaded successfully!


In [4]:
explainer = shap.TreeExplainer(
    xgb_ascentra
)

print("SHAP explainer ready!")

SHAP explainer ready!


In [5]:
def generate_student_risk_profile(student_row):

    # Remove ID and target
    X_student = student_row.drop(
        labels=["student_id", "academic_risk"],
        errors="ignore"
    ).to_frame().T

    # Preprocess
    X_processed = preprocessor.transform(X_student)

    # Prediction
    risk_probability = xgb_ascentra.predict_proba(
        X_processed
    )[0][1]

    prediction = xgb_ascentra.predict(
        X_processed
    )[0]

    # SHAP
    shap_values = explainer.shap_values(
        X_processed
    )

    if isinstance(shap_values, list):
        shap_values = shap_values[0]

    shap_values = np.asarray(shap_values).reshape(-1)

    # Feature names
    feature_names = (
        preprocessor.get_feature_names_out()
    )

    shap_df = pd.DataFrame({
        "Feature": feature_names,
        "SHAP Value": shap_values
    })

    shap_df["Absolute SHAP"] = (
        shap_df["SHAP Value"].abs()
    )

    shap_df = shap_df.sort_values(
        "Absolute SHAP",
        ascending=False
    )

    # Risk increasing
    risk_increasing = shap_df[
        shap_df["SHAP Value"] > 0
    ].head(5)

    # Risk reducing
    risk_reducing = shap_df[
        shap_df["SHAP Value"] < 0
    ].sort_values(
        "SHAP Value"
    ).head(5)

    # Risk level
    if risk_probability >= 0.70:
        risk_level = "HIGH"
    elif risk_probability >= 0.40:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"

    profile = {
        "student_id": student_row["student_id"],
        "risk_probability": round(
            float(risk_probability), 4
        ),
        "risk_percentage": round(
            float(risk_probability * 100), 2
        ),
        "risk_level": risk_level,
        "prediction": int(prediction),
        "risk_increasing_factors": risk_increasing,
        "risk_reducing_factors": risk_reducing
    }

    return profile

In [6]:
student = df.iloc[0]

profile = generate_student_risk_profile(
    student
)

print("===== ASCENTRA STUDENT RISK PROFILE =====")

print(
    "Student ID:",
    profile["student_id"]
)

print(
    "Risk Probability:",
    profile["risk_percentage"],
    "%"
)

print(
    "Risk Level:",
    profile["risk_level"]
)

===== ASCENTRA STUDENT RISK PROFILE =====
Student ID: STU0001
Risk Probability: 85.95 %
Risk Level: HIGH


In [7]:
print("\n===== RISK-INCREASING FACTORS =====")

display(
    profile["risk_increasing_factors"]
)


===== RISK-INCREASING FACTORS =====


,Feature,SHAP Value,Absolute SHAP
0,num__Curricular units 1st sem (approved),2.464418,2.464418
1,num__Curricular units 1st sem (grade),0.620584,0.620584
4,num__Admission grade,0.471704,0.471704
10,num__lms_login_frequency,0.351404,0.351404
5,num__attendance_percentage,0.342221,0.342221


In [8]:
print("\n===== RISK-REDUCING FACTORS =====")

display(
    profile["risk_reducing_factors"]
)


===== RISK-REDUCING FACTORS =====


,Feature,SHAP Value,Absolute SHAP
2,num__Curricular units 1st sem (enrolled),-2.038221,2.038221
3,num__Curricular units 1st sem (evaluations),-0.143105,0.143105
9,num__quiz_average,-0.062561,0.062561
6,num__ct1_score,-0.036469,0.036469
12,num__late_submission_count,-0.031871,0.031871


In [9]:
monitoring_features = [
    "attendance_percentage",
    "ct1_score",
    "ct2_score",
    "assignment_average",
    "quiz_average",
    "lms_login_frequency",
    "lms_resource_access",
    "late_submission_count",
    "academic_trend"
]

monitoring_data = student[
    monitoring_features
]

display(
    monitoring_data.to_frame(
        name="Value"
    )
)

,Value
attendance_percentage,54.4
ct1_score,10.4
ct2_score,52.9
assignment_average,40.0
quiz_average,38.9
lms_login_frequency,1.1
lms_resource_access,20.2
late_submission_count,7
academic_trend,Declining


In [10]:
student_profile = {
    "student_id": profile["student_id"],
    "risk_probability": profile["risk_percentage"],
    "risk_level": profile["risk_level"],

    "monitoring_data": monitoring_data.to_dict(),

    "risk_increasing_factors": (
        profile[
            "risk_increasing_factors"
        ][["Feature", "SHAP Value"]]
        .to_dict("records")
    ),

    "risk_reducing_factors": (
        profile[
            "risk_reducing_factors"
        ][["Feature", "SHAP Value"]]
        .to_dict("records")
    )
}

student_profile

{'student_id': 'STU0001',
 'risk_probability': 85.95,
 'risk_level': 'HIGH',
 'monitoring_data': {'attendance_percentage': 54.4,
  'ct1_score': 10.4,
  'ct2_score': 52.9,
  'assignment_average': 40.0,
  'quiz_average': 38.9,
  'lms_login_frequency': 1.1,
  'lms_resource_access': 20.2,
  'late_submission_count': 7,
  'academic_trend': 'Declining'},
 'risk_increasing_factors': [{'Feature': 'num__Curricular units 1st sem (approved)',
   'SHAP Value': 2.4644179344177246},
  {'Feature': 'num__Curricular units 1st sem (grade)',
   'SHAP Value': 0.6205844283103943},
  {'Feature': 'num__Admission grade', 'SHAP Value': 0.4717044234275818},
  {'Feature': 'num__lms_login_frequency', 'SHAP Value': 0.3514041602611542},
  {'Feature': 'num__attendance_percentage',
   'SHAP Value': 0.34222131967544556}],
 'risk_reducing_factors': [{'Feature': 'num__Curricular units 1st sem (enrolled)',
   'SHAP Value': -2.0382211208343506},
  {'Feature': 'num__Curricular units 1st sem (evaluations)',
   'SHAP Value': 